# Part 3 — Regression: Predicting `total_power`

**Goal:** Train a Linear Regression model to predict `total_power` from the
underlying Pokémon stats, validate it with a train/test split and
cross-validation, and report MAE, MSE, RMSE and R².


In [1]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("../models", exist_ok=True)
os.makedirs("../report", exist_ok=True)

df = pd.read_csv("../data/pokemon_feature_engineered.csv")
print("Loaded dataset:", df.shape)
df.head()


Loaded dataset: (200, 14)


## Feature / Target Setup

We predict `total_power` from the 9 underlying numeric attributes. Note that
`total_power` is *defined* as the exact sum of `hp + attack + defense +
special_attack + special_defense + speed` (per the Part 2 feature
engineering rule), so we expect Linear Regression to recover this exact
linear relationship with near-perfect accuracy — this is a useful sanity
check that the model correctly learns true linear structure in the data.

In [2]:
feature_cols = ["hp", "attack", "defense", "special_attack", "special_defense",
                "speed", "height", "weight", "base_experience"]
target_col = "total_power"

X = df[feature_cols]
y = df[target_col]

print("Features:", feature_cols)
print("Target:", target_col)


Features: ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed', 'height', 'weight', 'base_experience']
Target: total_power


## Train/Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape[0]} rows | Test size: {X_test.shape[0]} rows")


Train size: 160 rows | Test size: 40 rows


## Model Training

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Model Coefficients:")
for feat, coef in zip(feature_cols, model.coef_):
    print(f"  {feat:>16}: {coef:.4f}")
print(f"Intercept: {model.intercept_:.6f}")


Model Coefficients:
                hp: 1.0000
            attack: 1.0000
           defense: 1.0000
    special_attack: 1.0000
   special_defense: 1.0000
             speed: 1.0000
            height: -0.0000
            weight: -0.0000
   base_experience: -0.0000
Intercept: 0.000000


## Cross-Validation

In [5]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring="r2")

print("5-Fold Cross-Validation R\u00b2 scores:", np.round(cv_scores, 6))
print("Mean CV R\u00b2:", cv_scores.mean())


5-Fold Cross-Validation R² scores: [1. 1. 1. 1. 1.]
Mean CV R²: 1.0


## Test Set Evaluation

In [6]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

metrics = {
    "MAE": round(mae, 6),
    "MSE": round(mse, 6),
    "RMSE": round(rmse, 6),
    "R2_Score": round(r2, 6),
    "Mean_CV_R2": round(cv_scores.mean(), 6),
}

print("Test Set Metrics:")
for k, v in metrics.items():
    print(f"  {k:>12}: {v}")


Test Set Metrics:
           MAE: 0.0
           MSE: 0.0
          RMSE: 0.0
      R2_Score: 1.0
    Mean_CV_R2: 1.0


In [7]:
sample_compare = pd.DataFrame({
    "Actual": y_test.values[:10],
    "Predicted": np.round(y_pred[:10], 2)
})
print("Sample predictions vs actual (first 10 test rows):\n")
print(sample_compare.to_string(index=False))


Sample predictions vs actual (first 10 test rows):

 Actual  Predicted
    328      328.0
    251      251.0
    505      505.0
    405      405.0
    200      200.0
    295      295.0
    390      390.0
    460      460.0
    245      245.0
    285      285.0


## Save Metrics & Model

In [8]:
metrics_df = pd.DataFrame(list(metrics.items()), columns=["Metric", "Value"])
metrics_df.to_csv("../report/regression_metrics.csv", index=False)
print("Saved: ../report/regression_metrics.csv")
metrics_df


Saved: ../report/regression_metrics.csv


In [9]:
joblib.dump(model, "../models/linear_regression.pkl")
print("Saved: ../models/linear_regression.pkl")


Saved: ../models/linear_regression.pkl


## Summary

* **Linear Regression** achieves **R\u00b2 = 1.0** and **MAE/RMSE ≈ 0** on the
  held-out test set, with a 5-fold cross-validated R\u00b2 of 1.0 as well.
* This is expected, not a bug: `total_power` is *engineered* as an exact
  linear sum of six of the nine input features (`hp, attack, defense,
  special_attack, special_defense, speed`), so the regression coefficients
  converge to `1.0` for those six features and `\u22480` for the unrelated
  features (`height, weight, base_experience`), exactly recovering the
  ground-truth formula.
* This result demonstrates that **Linear Regression correctly identifies
  true linear relationships** in the data — a useful validation of both the
  modeling pipeline and the Part 2 feature engineering step.
* In a real-world setting (where the target is *not* a deterministic
  function of the features), we would expect R² well below 1.0; the
  takeaway here is about the model's ability to recover exact linear
  structure, not about overfitting.
